## Data import

In [ ]:
from datasets import load_dataset
import pandas as pd
import json, os

print("Downloading CICIDS2017...")
ds = load_dataset(
    "c01dsnap/CIC-IDS2017",
    data_files="Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    split="train"
)
df = ds.to_pandas()
df.columns = df.columns.str.strip()
print(f"Downloaded: {df.shape}, Labels: {df['Label'].value_counts().to_dict()}")

# ── 定義 label 的完整說明 ──────────────────────────────────────────
label_description = {
    "BENIGN":                   "benign",
    "Web Attack – Brute Force": "Web Attack – Brute Force",
    "Web Attack – XSS":         "Web Attack – XSS",
    "Web Attack – Sql Injection": "Web Attack – Sql Injection",
}

analyst_cols = [col for col in df.columns if col != "Label"]

# ── 平衡取樣（每個 label 最多 500 筆）──────────────────────────────
df_balanced = df.groupby("Label", group_keys=False).apply(
    lambda x: x.sample(min(len(x), 500), random_state=42)
)
print(f"平衡後：{df_balanced['Label'].value_counts().to_dict()}")

# ── 建立結構化 output（供 Final section 的 extract_label 正確解析）──
records = []
for _, row in df_balanced.iterrows():
    label = str(row["Label"]).strip()
    input_text = ", ".join([f"{col}: {row[col]}" for col in analyst_cols])
    readable_label = label_description.get(label, label)
    # 結構化格式：Classification 欄位讓 extract_label() 可正確解析
    output_text = (
        f"Summary: Network flow with {int(row.get('Total Fwd Packets', 0))} fwd pkts, "
        f"{int(row.get('Total Backward Packets', 0))} bwd pkts, duration {row.get('Flow Duration', 0)}.\n"
        f"Classification: {readable_label}\n"
        f"Confidence: 0.90\n"
        f"Key indicators: packet_count, flow_duration, flag_distribution"
    )
    records.append({
        "instruction": "You are a cybersecurity analyst. Analyze the following network flow and classify it.",
        "input": input_text,
        "output": output_text
    })

# ── 存檔 ────────────────────────────────────────────────────────────
os.makedirs("/LLaMA-Factory/data", exist_ok=True)
save_path = "/LLaMA-Factory/data/cicids_analyst_raw.json"
with open(save_path, "w") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print(f"\n✅ 完成！共 {len(records)} 筆")
print(f"儲存至：{save_path}")
print(f"\n第一筆 output 範例：\n{records[0]['output']}")


In [ ]:
import json

# [修正] 讀取 raw 資料（尚未做 train/test split）
with open("/LLaMA-Factory/data/cicids_analyst_raw.json") as f:
    data = json.load(f)

from collections import Counter
# [修正] 解析 Classification: 行來取得 label
labels = []
for d in data:
    for line in d["output"].splitlines():
        if line.strip().lower().startswith("classification"):
            labels.append(line.split(":", 1)[1].strip())
            break
print(Counter(labels))


In [ ]:
# 直接看 output 有哪些不同的開頭
import json
with open("/LLaMA-Factory/data/cicids_analyst_raw.json") as f:
    data = json.load(f)
outputs = set([d["output"][:80] for d in data])
for o in sorted(outputs):
    print(o)


In [ ]:
import json

dataset_info_path = "/LLaMA-Factory/data/dataset_info.json"

with open(dataset_info_path, "r", encoding="utf-8") as f:
    info = json.load(f)

# [修正] 登記 raw dataset（供預覽用，不作為訓練集）
# Final section 的 Cell 7 會正式登記帶 system prompt 的三個角色資料集
info["cicids_analyst_raw"] = {
    "file_name": "cicids_analyst_raw.json",
    "columns": {
        "prompt": "instruction",
        "query": "input",
        "response": "output"
    }
}

with open(dataset_info_path, "w", encoding="utf-8") as f:
    json.dump(info, f, ensure_ascii=False, indent=2)

print("✅ 登記成功！（cicids_analyst_raw）")


## Split dataset

In [ ]:
import json
import random
import math
import os

# ── 1. 讀取全部資料（來自 raw 檔）────────────────────────────────
# [修正] 讀取 cicids_analyst_raw.json 而非已存過的 cicids_analyst.json
with open("/LLaMA-Factory/data/cicids_analyst_raw.json", "r") as f:
    records = json.load(f)

print(f"總筆數: {len(records)}")

# ── 2. 打亂後按 80/20 比例切分 ────────────────────────────────────
# [修正] 改為比例切分，避免資料量不足 3000 時 test_data 為空
random.seed(42)
random.shuffle(records)

split_idx = math.ceil(len(records) * 0.8)
train_data = records[:split_idx]
test_data  = records[split_idx:]

print(f"Train: {len(train_data)} 筆")
print(f"Test:  {len(test_data)} 筆")
assert len(test_data) > 0, "❌ test_data 為空！請檢查資料量。"

# ── 3. 存 train ─────────────────────────────────────────────────
# [修正] 改名為 cicids_train.json，避免與 Final section 的角色檔案混淆
with open("/LLaMA-Factory/data/cicids_train.json", "w") as f:
    json.dump(train_data, f, ensure_ascii=False, indent=2)

# ── 4. 存 test ───────────────────────────────────────────────────
with open("/LLaMA-Factory/data/cicids_analyst_test.json", "w") as f:
    json.dump(test_data, f, ensure_ascii=False, indent=2)

# ── 5. 登記 test 到 dataset_info.json ────────────────────────────
info_path = "/LLaMA-Factory/data/dataset_info.json"
with open(info_path, "r") as f:
    info = json.load(f)

info["cicids_analyst_test"] = {
    "file_name": "cicids_analyst_test.json",
    "columns": {
        "prompt": "instruction",
        "query": "input",
        "response": "output"
    }
}

with open(info_path, "w") as f:
    json.dump(info, f, ensure_ascii=False, indent=2)

print("\n✅ 完成！")
print(f"- 訓練資料：cicids_train.json ({len(train_data)} 筆)")
print(f"- 測試資料：cicids_analyst_test.json ({len(test_data)} 筆)")
print("- dataset_info.json 已更新")


## Chat test

正常流量測試
測試 1 — BENIGN

Analyze the following network flow and classify it:
Flow Duration: 119261000, Total Fwd Packets: 20, Total Backward Packets: 18, Flow Bytes/s: 1024.5, Flow Packets/s: 45.2, SYN Flag Count: 1, ACK Flag Count: 35

攻擊流量測試
測試 2 — Web Attack – Brute Force

Analyze the following network flow and classify it:
Flow Duration: 302000, Total Fwd Packets: 8, Total Backward Packets: 6, Flow Bytes/s: 231.5, Flow Packets/s: 12.3, SYN Flag Count: 1, ACK Flag Count: 13, Destination Port: 80

測試 3 — Web Attack – XSS

Analyze the following network flow and classify it:
Flow Duration: 85000, Total Fwd Packets: 6, Total Backward Packets: 4, Flow Bytes/s: 980.0, Flow Packets/s: 21.5, SYN Flag Count: 1, ACK Flag Count: 9, Destination Port: 80

測試 4 — Web Attack – Sql Injection

Analyze the following network flow and classify it:
Flow Duration: 120000, Total Fwd Packets: 5, Total Backward Packets: 3, Flow Bytes/s: 640.0, Flow Packets/s: 14.2, SYN Flag Count: 1, ACK Flag Count: 7, Destination Port: 3306

（保留：補充說明）
原 測試5—Bot 與訓練資料 label 不符，已移除。訓練資料僅含 BENIGN、Web Attack–Brute Force、XSS、Sql Injection 四類。

**注意**：Chat test 的測試 prompt 應貼入 LLaMA Factory 或 Ollama 的 chat 介面，並設定 system prompt 為 SYSTEM_PROMPTS['analyst'] 的內容。

Final--三個角色

Cell 1

In [ ]:
import json
import random
import re
from pathlib import Path

SEED = 42
random.seed(SEED)

# ====== 修改這裡 ======
# [修正] INPUT_FILE 指向 cicids_train.json（Split 之後的訓練集），而非 cicids_analyst.json
INPUT_FILE = "/LLaMA-Factory/data/cicids_train.json"
OUTPUT_DIR = Path("/LLaMA-Factory/data")
# =====================

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# [新增] 直接在這裡載入 train_data，讓後續 Cell 5 可以使用
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    train_data = json.load(f)

print(f"✅ 輸出目錄：{OUTPUT_DIR}")
print(f"✅ 載入訓練資料：{len(train_data)} 筆")


Cell 2

In [ ]:
SYSTEM_PROMPTS = {
    "analyst": (
        "You are the Analyst Agent in an AI Parliament for cybersecurity. "
        "Your role: given raw network flow features, produce a structured event summary "
        "and an initial traffic classification. Be factual and concise. "
        "Output format:\n"
        "Summary: <one-sentence description of the traffic pattern>\n"
        "Classification: <label>\n"
        "Confidence: <0.0–1.0>\n"
        "Key indicators: <comma-separated list>"
    ),
    "hunter": (
        "You are the Hunter Agent in an AI Parliament for cybersecurity. "
        "Your role: given the Analyst's event summary and classification, "
        "map the attack to MITRE ATT&CK techniques, estimate the threat level, "
        "and propose an attack hypothesis. "
        "Output format:\n"
        "ATT&CK Technique: <TID> – <name>\n"
        "Attack Stage: <Reconnaissance|Initial Access|Execution|…>\n"
        "Threat Level: <Low|Medium|High|Critical>\n"
        "Hypothesis: <2-sentence narrative>\n"
        "Confidence: <0.0–1.0>"
    ),
    "auditor": (
        "You are the Auditor Agent in an AI Parliament for cybersecurity. "
        "Your role: critically review the Analyst and Hunter outputs. "
        "Identify logical gaps, overly confident claims, or alternative explanations. "
        "Output format:\n"
        "Agreed: <what you agree with>\n"
        "Challenged: <what you dispute and why>\n"
        "Alternative hypothesis: <if any, else 'None'>\n"
        "Adjusted confidence: <0.0–1.0>\n"
        "Recommendation: <suggested next investigative step>"
    ),
}

print("✅ System prompts 載入完成")

Cell 3

In [ ]:
ATTACK_MAPPING = {
    "DDoS":                           ("T1498",     "Network Denial of Service",              "Impact",           "Critical"),
    "DoS Hulk":                       ("T1499",     "Endpoint Denial of Service",             "Impact",           "High"),
    "DoS SlowHTTPTest":               ("T1499.001", "OS Exhaustion Flood",                    "Impact",           "Medium"),
    "DoS Slowloris":                  ("T1499.001", "OS Exhaustion Flood",                    "Impact",           "Medium"),
    "DoS GoldenEye":                  ("T1499.002", "Service Exhaustion Flood",               "Impact",           "High"),
    "DoS SlowMoSh":                   ("T1499.001", "OS Exhaustion Flood",                    "Impact",           "Medium"),
    "PortScan":                       ("T1046",     "Network Service Discovery",              "Discovery",        "Medium"),
    "FTP-Patator":                    ("T1110",     "Brute Force",                            "Credential Access","High"),
    "SSH-Patator":                    ("T1110.001", "Password Guessing",                      "Credential Access","High"),
    "Bot":                            ("T1071",     "Application Layer Protocol",             "Command and Control","High"),
    "Infiltration":                   ("T1190",     "Exploit Public-Facing Application",      "Initial Access",   "Critical"),
    "Web Attack – Brute Force":       ("T1110",     "Brute Force",                            "Credential Access","High"),
    "Web Attack – XSS":               ("T1059.007", "JavaScript",                             "Execution",        "Medium"),
    "Web Attack – Sql Injection":     ("T1190",     "Exploit Public-Facing Application",      "Initial Access",   "Critical"),
    "Heartbleed":                     ("T1212",     "Exploitation for Credential Access",     "Credential Access","Critical"),
    "BENIGN":                         ("N/A",       "N/A",                                    "N/A",              "None"),
}

def extract_label(output_text: str) -> str:
    for line in output_text.splitlines():
        if line.strip().lower().startswith("classification"):
            parts = line.split(":", 1)
            if len(parts) == 2:
                return parts[1].strip()
    return output_text.strip().splitlines()[0]

def extract_confidence(output_text: str) -> float:
    for line in output_text.splitlines():
        lower = line.strip().lower()
        if lower.startswith("confidence") or lower.startswith("adjusted confidence"):
            parts = line.split(":", 1)
            if len(parts) == 2:
                try:
                    return float(parts[1].strip())
                except ValueError:
                    pass
    return round(random.uniform(0.6, 0.9), 2)

def get_attack_info(label: str):
    for key in ATTACK_MAPPING:
        if key.lower() in label.lower():
            return ATTACK_MAPPING[key]
    return ("T0000", "Unknown Technique", "Unknown Stage", "Unknown")

print("✅ ATT&CK 對應表 & 工具函數載入完成")

Cell 4

In [ ]:
def build_hunter_output(analyst_output: str, label: str) -> str:
    tid, tname, stage, threat = get_attack_info(label)
    conf = round(random.uniform(0.55, 0.92), 2)
    if label.upper() == "BENIGN":
        hypothesis = (
            "The traffic pattern appears consistent with legitimate usage. "
            "No malicious indicators were identified in the available features."
        )
    else:
        hypothesis = (
            f"Based on the Analyst's summary, this traffic is consistent with a {label} attack. "
            f"The attacker likely used {tname} ({tid}) during the {stage} phase to achieve their objective."
        )
    return (
        f"ATT&CK Technique: {tid} – {tname}\n"
        f"Attack Stage: {stage}\n"
        f"Threat Level: {threat}\n"
        f"Hypothesis: {hypothesis}\n"
        f"Confidence: {conf}"
    )

def build_auditor_output(analyst_output: str, hunter_output: str, label: str) -> str:
    hunter_conf = extract_confidence(hunter_output)
    adjusted    = round(max(0.1, hunter_conf - random.uniform(0.0, 0.15)), 2)

    if label.upper() == "BENIGN":
        agreed     = "The traffic is consistent with normal behavior."
        challenged = (
            "Ensure no low-and-slow attack patterns are masked by normal averages. "
            "ACK/SYN ratio should be verified across a longer time window."
        )
        alt_hypo   = "Possible port-scan or reconnaissance with very low packet rate."
        rec        = "Monitor the same source IP over the next 30-minute window for anomaly escalation."
    else:
        agreed     = f"The {label} classification aligns with the anomalous features reported by the Analyst."
        challenged = (
            "The confidence score may be inflated. "
            "Consider whether the observed byte/packet rates could also match a legitimate high-throughput service."
        )
        alt_hypo   = "Possible misconfigured client causing burst traffic rather than deliberate attack."
        rec        = "Correlate with endpoint logs to confirm malicious intent before escalating to incident response."

    return (
        f"Agreed: {agreed}\n"
        f"Challenged: {challenged}\n"
        f"Alternative hypothesis: {alt_hypo}\n"
        f"Adjusted confidence: {adjusted}\n"
        f"Recommendation: {rec}"
    )

print("✅ Hunter & Auditor 函數載入完成")

Cell 5

In [ ]:
analyst_data = []
hunter_data  = []
auditor_data = []

for item in train_data:
    instruction    = item.get("instruction", "")
    traffic_input  = item.get("input", "")       # [修正] 實際流量特徵
    analyst_output = item.get("output", "")
    label          = extract_label(analyst_output)

    # --- Analyst ---
    analyst_data.append({
        "system":      SYSTEM_PROMPTS["analyst"],
        "instruction": instruction,
        "input":       traffic_input,
        "output":      analyst_output,
    })

    # --- Hunter ---
    # [修正] [Traffic Features] 使用 traffic_input（實際特徵值），而非 instruction（角色說明）
    hunter_instruction = (
        f"The Analyst has reviewed the following network traffic:\n"
        f"[Traffic Features]\n{traffic_input}\n\n"
        f"[Analyst Output]\n{analyst_output}\n\n"
        f"Now perform threat hunting and map to MITRE ATT&CK."
    )
    hunter_output = build_hunter_output(analyst_output, label)
    hunter_data.append({
        "system":      SYSTEM_PROMPTS["hunter"],
        "instruction": hunter_instruction,
        "input":       "",
        "output":      hunter_output,
    })

    # --- Auditor ---
    # [修正] [Traffic Features] 同樣使用 traffic_input
    auditor_instruction = (
        f"Review the following multi-agent outputs and critically evaluate them:\n\n"
        f"[Traffic Features]\n{traffic_input}\n\n"
        f"[Analyst Output]\n{analyst_output}\n\n"
        f"[Hunter Output]\n{hunter_output}\n\n"
        f"Provide a critical audit."
    )
    auditor_output = build_auditor_output(analyst_output, hunter_output, label)
    auditor_data.append({
        "system":      SYSTEM_PROMPTS["auditor"],
        "instruction": auditor_instruction,
        "input":       "",
        "output":      auditor_output,
    })

print(f"✅ Analyst : {len(analyst_data)} 筆")
print(f"✅ Hunter  : {len(hunter_data)} 筆")
print(f"✅ Auditor : {len(auditor_data)} 筆")


Cell 6

In [ ]:
# [確認] Cell 6 儲存三個角色資料集
# cicids_analyst.json 此時存的是帶 system prompt 的 analyst_data，
# dataset_info.json 會在 Cell 7 正式更新，新增 system 欄位登記
for role, data in [("analyst", analyst_data), ("hunter", hunter_data), ("auditor", auditor_data)]:
    path = OUTPUT_DIR / f"cicids_{role}.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"✅ 儲存 cicids_{role}.json → {path}  ({len(data)} 筆)")


Cell 7

In [ ]:
dataset_info_path = OUTPUT_DIR / "dataset_info.json"

with open(dataset_info_path, "r") as f:
    dataset_info = json.load(f)

# [修正] 三個角色資料集都帶有 system 欄位
# cicids_analyst_raw 和 cicids_analyst_test 已在前面 section 登記，這裡只更新/新增角色集
new_entries = {
    "cicids_analyst": {
        "file_name": "cicids_analyst.json",
        "columns": {"prompt": "instruction", "query": "input", "response": "output", "system": "system"}
    },
    "cicids_hunter": {
        "file_name": "cicids_hunter.json",
        "columns": {"prompt": "instruction", "query": "input", "response": "output", "system": "system"}
    },
    "cicids_auditor": {
        "file_name": "cicids_auditor.json",
        "columns": {"prompt": "instruction", "query": "input", "response": "output", "system": "system"}
    },
}

dataset_info.update(new_entries)

with open(dataset_info_path, "w", encoding="utf-8") as f:
    json.dump(dataset_info, f, ensure_ascii=False, indent=2)

print("✅ dataset_info.json 更新完成，已登記（含 system 欄位）：")
for k in new_entries:
    print(f"   - {k}")
print("\n⚠️  LLaMA Factory 訓練時請分別針對 cicids_analyst / cicids_hunter / cicids_auditor 各訓練一個 LoRA adapter。")


Cell 8

In [ ]:
from collections import Counter

# [修正] analyst_data 的 output 已是結構化格式，extract_label 可正確解析
label_counts = Counter(extract_label(d["output"]) for d in analyst_data)
print("📊 各類別筆數（Analyst 訓練資料集）：")
max_count = max(label_counts.values()) if label_counts else 1
for label, count in sorted(label_counts.items(), key=lambda x: -x[1]):
    bar = "█" * (count * 30 // max_count)
    print(f"  {label:<35} {count:>5}  {bar}")

# 驗證：確認所有 label 都在 ATTACK_MAPPING 中
print("\n🔍 ATTACK_MAPPING 覆蓋率：")
all_mapped = True
for label in label_counts:
    found = any(k.lower() in label.lower() for k in ATTACK_MAPPING)
    status = "✅" if found else "⚠️  未對應"
    if not found:
        all_mapped = False
    print(f"  {status}  {label}")
if all_mapped:
    print("\n✅ 所有 label 已對應 ATTACK_MAPPING")
else:
    print("\n⚠️  部分 label 未對應，會使用 Unknown Technique 填補")


## LLaMA Factory 訓練指令

完成資料準備後，分別對三個角色資料集執行 LoRA fine-tuning：

```bash
# Analyst agent
llamafactory-cli train \
  --model_name_or_path meta-llama/Llama-3.2-3B-Instruct \
  --dataset cicids_analyst \
  --template llama3 \
  --finetuning_type lora \
  --output_dir /LLaMA-Factory/saves/analyst \
  --num_train_epochs 3 --per_device_train_batch_size 4 --lora_rank 16

# Hunter agent（依此類推 cicids_hunter, cicids_auditor）
```

**驗證**：訓練完成後使用 LLaMA Factory WebUI 的 Chat 功能，分別載入三個 adapter，貼入 Chat test section 的測試 prompt 驗證輸出格式。

Auditor

Cell A

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL = "meta-llama/Llama-3.1-8B-Chinese-Chat"

def load_agent(adapter_path):
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, torch_dtype=torch.float16, device_map="auto"
    )
    model = PeftModel.from_pretrained(model, adapter_path)
    model.eval()
    return model, tokenizer

def infer(model, tokenizer, system_prompt, user_input, max_new_tokens=256):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_input},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
        )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

Cell B

In [ ]:
print("Loading Analyst...")
analyst_model, analyst_tok = load_agent("LLaMA-Factory/saves/Llama-3.1-8B-Chinese-Chat/lora/train_2026-06-10-09-36-17")

print("Loading Hunter...")
hunter_model, hunter_tok = load_agent("LLaMA-Factory/saves/Llama-3.1-8B-Chinese-Chat/lora/hunter")

print("Loading Auditor...")
auditor_model, auditor_tok = load_agent("LLaMA-Factory/saves/Llama-3.1-8B-Chinese-Chat/lora/Auditor")

print("✅ 三個 agent 載入完成")

Cell C

In [ ]:
def parliament_judge(traffic_features: str, verbose=False):
    # ── Step 1: Analyst ──────────────────────────────────────
    analyst_out = infer(
        analyst_model, analyst_tok,
        system_prompt=SYSTEM_PROMPTS["analyst"],
        user_input=traffic_features
    )
    if verbose:
        print("=== Analyst ===\n", analyst_out, "\n")

    # ── Step 2: Hunter ──────────────────────────────────────
    hunter_input = (
        f"[Traffic Features]\n{traffic_features}\n\n"
        f"[Analyst Output]\n{analyst_out}\n\n"
        f"Now perform threat hunting and map to MITRE ATT&CK."
    )
    hunter_out = infer(
        hunter_model, hunter_tok,
        system_prompt=SYSTEM_PROMPTS["hunter"],
        user_input=hunter_input
    )
    if verbose:
        print("=== Hunter ===\n", hunter_out, "\n")

    # ── Step 3: Auditor ─────────────────────────────────────
    auditor_input = (
        f"[Traffic Features]\n{traffic_features}\n\n"
        f"[Analyst Output]\n{analyst_out}\n\n"
        f"[Hunter Output]\n{hunter_out}\n\n"
        f"Provide a critical audit."
    )
    auditor_out = infer(
        auditor_model, auditor_tok,
        system_prompt=SYSTEM_PROMPTS["auditor"],
        user_input=auditor_input
    )

    print("=" * 50)
    print("🏛️  AI Parliament Final Verdict")
    print("=" * 50)
    print(auditor_out)
    return auditor_out

Cell D

In [ ]:
# 正常流量測試 測試 1 — BENIGN
parliament_judge("""
Analyze the following network flow and classify it: Flow Duration: 119261000, Total Fwd Packets: 20, Total Backward Packets: 18, Flow Bytes/s: 1024.5, Flow Packets/s: 45.2, SYN Flag Count: 1, ACK Flag Count: 35
""")
#攻擊流量測試 測試 2 — Web Attack – Brute Force
parliament_judge("""
Analyze the following network flow and classify it: Flow Duration: 302000, Total Fwd Packets: 8, Total Backward Packets: 6, Flow Bytes/s: 231.5, Flow Packets/s: 12.3, SYN Flag Count: 1, ACK Flag Count: 13, Destination Port: 80
""")
#測試 3 — Web Attack – XSS
parliament_judge("""
Analyze the following network flow and classify it: Flow Duration: 85000, Total Fwd Packets: 6, Total Backward Packets: 4, Flow Bytes/s: 980.0, Flow Packets/s: 21.5, SYN Flag Count: 1, ACK Flag Count: 9, Destination Port: 80
""")
#測試 4 — Web Attack – Sql Injection
parliament_judge("""
Analyze the following network flow and classify it: Flow Duration: 120000, Total Fwd Packets: 5, Total Backward Packets: 3, Flow Bytes/s: 640.0, Flow Packets/s: 14.2, SYN Flag Count: 1, ACK Flag Count: 7, Destination Port: 3306
""")
#測試 5 — Bot
parliament_judge("""
Analyze the following network traffic and classify it: Flow Duration: 7200000000, Total Fwd Packets: 3, Total Backward Packets: 2, Flow Bytes/s: 0.8, Flow Packets/s: 0.0007, SYN Flag Count: 1, ACK Flag Count: 4, Destination Port: 6667
""")

In [ ]:
parliament_judge("...", verbose=True)

量化評估

In [ ]:
!pip install rouge-score

In [ ]:
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
import json, time, nltk
nltk.download('punkt_tab', quiet=True)

with open("/LLaMA-Factory/data/cicids_analyst_test.json") as f:
    test_data = json.load(f)

rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
scores = {"rouge1": [], "rouge2": [], "rougeL": []}
references_bleu, hypotheses_bleu = [], []

start_time = time.time()  # ← 開始計時

for i, item in enumerate(test_data[:50]):
    traffic   = item["input"]
    reference = item["output"]

    prediction = parliament_judge(traffic, verbose=False)

    # ROUGE
    r = rouge.score(reference, prediction)
    scores["rouge1"].append(r["rouge1"].fmeasure)
    scores["rouge2"].append(r["rouge2"].fmeasure)
    scores["rougeL"].append(r["rougeL"].fmeasure)

    # BLEU
    references_bleu.append([reference.split()])
    hypotheses_bleu.append(prediction.split())

    if i % 10 == 0:
        print(f"進度: {i+1}/50")

total_time = time.time() - start_time  # ← 總執行時間
n_samples  = len(test_data[:50])

# 計算 BLEU-4
smooth = SmoothingFunction().method1
bleu4  = corpus_bleu(references_bleu, hypotheses_bleu,
                     weights=(0.25, 0.25, 0.25, 0.25),
                     smoothing_function=smooth) * 100

print("\n=== 🏛 AI Parliament 完整評估結果 ===")
print(f"predict_bleu-4              : {bleu4:.2f}")
print(f"predict_rouge-1             : {sum(scores['rouge1'])/n_samples*100:.2f}")
print(f"predict_rouge-2             : {sum(scores['rouge2'])/n_samples*100:.2f}")
print(f"predict_rouge-l             : {sum(scores['rougeL'])/n_samples*100:.2f}")
print(f"predict_runtime             : {total_time:.1f}s")
print(f"predict_samples_per_second  : {n_samples/total_time:.2f}")